# SRC-001 Lower-Layer Database Audit

## Purpose

Inspect the structure, content, quality, and temporal characteristics of the
lower-layer telemetry database contained in SRC-001.

## Out of Scope

This notebook does not:

- modify or clean the database;
- analyze PCAP files;
- analyze scenario-level TXT files;
- perform cross-source alignment;
- evaluate RCA performance.

## 1. Setup

Locate the lower-layer SQLite database associated with SRC-001 and verify that
it can be opened successfully before inspecting its internal structure.

The database is discovered from the raw source directory rather than referenced
through a machine-specific absolute path.

In [1]:
from src.data import inventory

import sqlite3

import pandas as pd
import numpy as np

In [2]:
SOURCE_ID = "SRC-001"

inventory = inventory.DataInventory(source_id=SOURCE_ID)

RAW_DATA_PATH = inventory.raw_data_path

In [3]:
db_files = sorted(
    path
    for path in RAW_DATA_PATH.iterdir()
    if path.is_file() and path.suffix.lower() == ".db"
)

db_files

[PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/Lower_Layer_Data.db')]

In [4]:
if not db_files:
    raise FileNotFoundError(
        f"No database files found in: {RAW_DATA_PATH}"
    )

In [5]:
for path in db_files:
    print(path.name)

Lower_Layer_Data.db


In [6]:
LOWER_LAYER_DB_PATH = db_files[0]

LOWER_LAYER_DB_PATH

PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/Lower_Layer_Data.db')

In [7]:
with sqlite3.connect(LOWER_LAYER_DB_PATH) as connection:
    connection.execute("SELECT 1")

print("Database readable:", LOWER_LAYER_DB_PATH.name)

Database readable: Lower_Layer_Data.db


### Finding

A single SQLite database was identified in the SRC-001 raw data directory:
`Lower_Layer_Data.db`.

The database can be opened successfully through SQLite, confirming that the
file is readable and available for structural inspection.

No database contents have been modified or analyzed at this stage.

## 2. Database Structure

Inspect the internal SQLite structure before analyzing telemetry values.

This section identifies available tables, their schemas, and record counts.

In [8]:
with sqlite3.connect(LOWER_LAYER_DB_PATH) as connection:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
        """,
        connection,
    )

tables

,name
0,lower_layer_data


In [9]:
table_schemas = {}

with sqlite3.connect(LOWER_LAYER_DB_PATH) as connection:
    for table in tables["name"]:
        schema = pd.read_sql_query(
            f"PRAGMA table_info('{table}')",
            connection,
        )

        table_schemas[table] = schema

        print(f"\nTable: {table}")
        display(schema)


Table: lower_layer_data


,cid,name,type,notnull,dflt_value,pk
0,0,dlBytes,INTEGER,0,None,0
1,1,dlMcs,INTEGER,0,None,0
2,2,dlBler,REAL,0,None,0
3,3,ulBytes,INTEGER,0,None,0
4,4,ulMcs,INTEGER,0,None,0
5,5,ulBler,REAL,0,None,0
6,6,ri,INTEGER,0,None,0
7,7,phr,INTEGER,0,None,0
8,8,pcmax,INTEGER,0,None,0
9,9,rsrq,REAL,0,None,0


In [10]:
record_counts = []

with sqlite3.connect(LOWER_LAYER_DB_PATH) as connection:
    for table in tables["name"]:
        count = connection.execute(
            f'SELECT COUNT(*) FROM "{table}"'
        ).fetchone()[0]

        record_counts.append(
            {
                "table": table,
                "record_count": count,
            }
        )

record_counts = pd.DataFrame(record_counts)

record_counts

,table,record_count
0,lower_layer_data,45244


### Finding

`Lower_Layer_Data.db` is readable and contains a single table,
`lower_layer_data`.

The table contains 25 fields covering lower-layer telemetry measurements,
radio-quality indicators, UE/cell identifiers, synchronization-related fields,
a timestamp field, and ground-truth labels including `traffic_type`,
`attack_category`, and `attack_subcategory`.

A total of 45,244 records are stored in the table.

The database structure is therefore simple, self-contained, and suitable for
direct content-level auditing in the following sections.

## 3. Telemetry Structure

Inspect the fields, data types, and cardinality of the lower-layer telemetry
records to understand the information represented by the dataset.

In [11]:
TABLE_NAME = tables["name"].iloc[0]

with sqlite3.connect(LOWER_LAYER_DB_PATH) as connection:
    telemetry = pd.read_sql_query(
        f'SELECT * FROM "{TABLE_NAME}"',
        connection,
    )

telemetry.head()

,dlBytes,dlMcs,dlBler,ulBytes,ulMcs,ulBler,ri,phr,pcmax,rsrq,...,puschSnr,ue_id,timestamp,cellid,in_sync,rnti,pmi,traffic_type,attack_category,attack_subcategory
0,93883433,13,0.0,115160671,9,0.057268,1,15,24,-11.0,...,33.0,1.0,1.751034e+12,11111111.0,1.0,NaN,"(0, 0)",1,DoS,DoS_SYN
1,93888597,11,0.0,115199297,10,0.109026,1,11,24,-11.0,...,19.0,1.0,1.751360e+12,11111111.0,1.0,NaN,"(0, 0)",1,DoS,DoS_SYN
2,93889971,11,0.0,115202820,9,0.038015,1,11,24,-11.0,...,35.5,1.0,1.751360e+12,11111111.0,1.0,NaN,"(0, 0)",1,DoS,DoS_SYN
3,93894248,9,0.0,115248349,12,0.018447,1,13,23,-11.0,...,32.0,1.0,1.751037e+12,11111111.0,1.0,NaN,"(0, 0)",1,DoS,DoS_SYN
4,93898044,10,0.0,115305003,11,0.007147,1,19,23,-11.0,...,10.0,1.0,1.751037e+12,11111111.0,1.0,NaN,"(0, 0)",1,DoS,DoS_SYN


In [12]:
telemetry.shape

(45244, 25)

In [13]:
field_summary = pd.DataFrame(
    {
        "field": telemetry.columns,
        "dtype": telemetry.dtypes.astype(str).values,
    }
)

field_summary

,field,dtype
0,dlBytes,int64
1,dlMcs,int64
2,dlBler,float64
3,ulBytes,int64
4,ulMcs,int64
5,ulBler,float64
6,ri,int64
7,phr,int64
8,pcmax,int64
9,rsrq,float64


In [14]:
cardinality_summary = pd.DataFrame(
    {
        "field": telemetry.columns,
        "unique_values": [
            telemetry[column].nunique(dropna=False)
            for column in telemetry.columns
        ],
    }
).sort_values(
    "unique_values",
    ascending=False,
).reset_index(drop=True)

cardinality_summary

,field,unique_values
0,dlBytes,44737
1,ulBytes,43888
2,timestamp,31550
3,ulBler,30258
4,dlBler,28623
5,rssi,157
6,puschSnr,87
7,sinr,61
8,rnti,46
9,pucchSnr,35


### Finding

The lower-layer dataset contains 45,244 telemetry records across 25 fields.

Most telemetry fields are numeric and represent radio or traffic measurements,
while categorical fields include identifiers and ground-truth labels.

The dataset contains:

- 9 distinct `ue_id` values;
- 46 distinct `rnti` values;
- 6 attack categories;
- 19 attack subcategories;
- 2 traffic classes;
- an explicit `timestamp` field with 31,550 distinct values.

Four fields (`cellid`, `in_sync`, `pmi`, and `ri`) contain only one distinct
value and require further evaluation as potential constant fields.

The structure is suitable for proceeding with data-quality validation.

## 4. Data Quality

Evaluate missing values, duplicate observations, constant fields, and
potentially invalid telemetry values.

In [15]:
missing_summary = pd.DataFrame(
    {
        "field": telemetry.columns,
        "missing_count": telemetry.isna().sum().values,
        "missing_pct": (
            telemetry.isna().mean().values * 100
        ),
    }
).sort_values(
    "missing_count",
    ascending=False,
).reset_index(drop=True)

missing_summary

,field,missing_count,missing_pct
0,rnti,1540,3.403766
1,dlBytes,0,0.000000
2,cqi,0,0.000000
3,attack_category,0,0.000000
4,traffic_type,0,0.000000
5,pmi,0,0.000000
6,in_sync,0,0.000000
7,cellid,0,0.000000
8,timestamp,0,0.000000
9,ue_id,0,0.000000


In [16]:
missing_summary[
    missing_summary["missing_count"] > 0
]

,field,missing_count,missing_pct
0,rnti,1540,3.403766


In [17]:
duplicate_mask = telemetry.duplicated(keep="first")

duplicate_count = duplicate_mask.sum()
duplicate_pct = duplicate_count / len(telemetry) * 100

print("Total records:", len(telemetry))
print("Duplicate records:", duplicate_count)
print(f"Duplicate percentage: {duplicate_pct:.4f}%")

Total records: 45244
Duplicate records: 0
Duplicate percentage: 0.0000%


In [18]:
duplicate_examples = telemetry[
    telemetry.duplicated(keep=False)
].head(20)

duplicate_examples

,dlBytes,dlMcs,dlBler,ulBytes,ulMcs,ulBler,ri,phr,pcmax,rsrq,...,puschSnr,ue_id,timestamp,cellid,in_sync,rnti,pmi,traffic_type,attack_category,attack_subcategory


In [19]:
constant_fields = [
    column
    for column in telemetry.columns
    if telemetry[column].nunique(dropna=False) <= 1
]

constant_fields

['ri', 'cellid', 'in_sync', 'pmi']

In [20]:
numeric_fields = telemetry.select_dtypes(include="number").columns

infinite_counts = {
    column: np.isinf(telemetry[column]).sum()
    for column in numeric_fields
}

{
    column: count
    for column, count in infinite_counts.items()
    if count > 0
}

{}

In [21]:
byte_fields = [
    column
    for column in telemetry.columns
    if "bytes" in column.lower()
]

negative_byte_counts = {
    column: (telemetry[column] < 0).sum()
    for column in byte_fields
}

negative_byte_counts

{'dlBytes': np.int64(0), 'ulBytes': np.int64(0)}

In [22]:
binary_fields = [
    column
    for column in numeric_fields
    if set(telemetry[column].dropna().unique()).issubset({0, 1})
]

binary_fields

['ri', 'in_sync', 'traffic_type']

In [23]:
timestamp_invalid = (
    telemetry["timestamp"].isna()
    | ~np.isfinite(telemetry["timestamp"])
    | (telemetry["timestamp"] <= 0)
)

print("Invalid timestamps:", timestamp_invalid.sum())

Invalid timestamps: 0


### Finding

The lower-layer telemetry dataset shows generally good structural quality.

Only `rnti` contains missing values, with 1,540 missing records
(approximately 3.40% of the dataset).

No exact duplicate records were detected.

Four constant fields were identified: `ri`, `cellid`, `in_sync`, and `pmi`.
These fields provide no variation within the current SRC-001 subset and should
be treated carefully in later analysis.

No infinite values or negative byte counters were detected.

The binary-valued fields identified in the dataset are `ri`, `in_sync`, and
`traffic_type`, and no structurally invalid timestamp values were found.

Overall, the dataset is structurally usable, with the missing `rnti` values and
constant fields representing the main quality limitations identified at this
stage.

## 5. Ground Truth and Scenario Structure

Verify the ground-truth fields, their distribution, and the consistency of
scenario labels in the lower-layer telemetry dataset.

In [24]:
ground_truth_fields = [
    "traffic_type",
    "attack_category",
    "attack_subcategory",
]

missing_ground_truth_fields = [
    column
    for column in ground_truth_fields
    if column not in telemetry.columns
]

missing_ground_truth_fields

[]

In [25]:
label_distributions = {}

for column in ground_truth_fields:
    summary = (
        telemetry[column]
        .value_counts(dropna=False)
        .rename("count")
        .reset_index()
    )

    summary["percentage"] = (
        summary["count"] / len(telemetry) * 100
    )

    label_distributions[column] = summary

    print(f"\n{column}")
    display(summary)


traffic_type


,traffic_type,count,percentage
0,1,34580,76.430024
1,0,10664,23.569976



attack_category


,attack_category,count,percentage
0,DDoS,15084,33.339227
1,Benign,10664,23.569976
2,Probe,8445,18.665458
3,DoS,4721,10.434533
4,Web Attacks,3273,7.234108
5,BruteForce,3057,6.756697



attack_subcategory


,attack_subcategory,count,percentage
0,Benign,10664,23.569976
1,Probe_OsFingerprinting,6036,13.340995
2,DDoS_UDP,4458,9.853240
3,DDoS_SYN,3568,7.886129
4,DDoS_ICMP,3294,7.280523
5,Probe_PortScan,2409,5.324463
6,DDoS_ACK,1934,4.274600
7,DDoS_Slowloris,1830,4.044735
8,BruteForce_SSH,1666,3.682256
9,BruteForce_FTP,1391,3.074441


In [26]:
scenario_structure = (
    telemetry[
        [
            "traffic_type",
            "attack_category",
            "attack_subcategory",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "traffic_type",
            "attack_category",
            "attack_subcategory",
        ]
    )
    .reset_index(drop=True)
)

scenario_structure

,traffic_type,attack_category,attack_subcategory
0,0,Benign,Benign
1,1,BruteForce,BruteForce_FTP
2,1,BruteForce,BruteForce_SSH
3,1,DDoS,DDoS_ACK
4,1,DDoS,DDoS_ICMP
5,1,DDoS,DDoS_SYN
6,1,DDoS,DDoS_Slowloris
7,1,DDoS,DDoS_UDP
8,1,DoS,DoS_ACK
9,1,DoS,DoS_ICMP


In [27]:
invalid_scenarios = telemetry[
    (
        (telemetry["traffic_type"] == 0)
        & (
            (telemetry["attack_category"] != "benign")
            | (telemetry["attack_subcategory"] != "benign")
        )
    )
    |
    (
        (telemetry["traffic_type"] == 1)
        & (
            (telemetry["attack_category"] == "benign")
            | (telemetry["attack_subcategory"] == "benign")
        )
    )
]

len(invalid_scenarios)

10664

### Finding

The lower-layer telemetry dataset contains three hierarchical ground-truth
fields: `traffic_type`, `attack_category`, and `attack_subcategory`.

Attack traffic represents approximately 76.43% of the dataset, while benign
traffic represents approximately 23.57%.

Six high-level categories and 19 subcategories are represented. DDoS is the
largest attack category, followed by Probe, DoS, Web Attacks, and BruteForce.

The observed scenario hierarchy is coherent: benign records are consistently
represented by the benign category and subcategory, while attack records are
associated with their corresponding attack scenarios.

The dataset therefore provides structured ground-truth information suitable
for scenario-level analysis and evaluation.

## 6. Temporal Structure

Evaluate whether lower-layer telemetry preserves valid temporal information and
determine the time period covered by the dataset.

In [30]:
temporal_fields = [
    column
    for column in telemetry.columns
    if any(token in column.lower() for token in ("timestamp", "time", "date"))
]

temporal_fields

['timestamp']

In [31]:
temporal_summary = telemetry[temporal_fields].agg(
    ["count", "nunique", "min", "max"]
)

temporal_summary

,timestamp
count,4.524400e+04
nunique,3.155000e+04
min,1.746800e+12
max,1.751375e+12


In [32]:
timestamp_validity = pd.DataFrame(
    {
        "field": temporal_fields,
        "missing": [
            telemetry[column].isna().sum()
            for column in temporal_fields
        ],
        "non_positive": [
            (telemetry[column] <= 0).sum()
            for column in temporal_fields
        ],
        "infinite": [
            (~np.isfinite(telemetry[column])).sum()
            for column in temporal_fields
        ],
    }
)

timestamp_validity

,field,missing,non_positive,infinite
0,timestamp,0,0,0


In [33]:
for column in temporal_fields:
    print(column)
    print("Min:", telemetry[column].min())
    print("Max:", telemetry[column].max())

timestamp
Min: 1746799506701.0
Max: 1751375357835.0


In [34]:
TIMESTAMP_FIELD = temporal_fields[0]

telemetry["timestamp_datetime"] = pd.to_datetime(
    telemetry[TIMESTAMP_FIELD],
    unit="ms",
    utc=True,
)

print("Start:", telemetry["timestamp_datetime"].min())
print("End:", telemetry["timestamp_datetime"].max())

Start: 2025-05-09 14:05:06.701000+00:00
End: 2025-07-01 13:09:17.835000+00:00


In [35]:
temporal_coverage = (
    telemetry["timestamp_datetime"].max()
    - telemetry["timestamp_datetime"].min()
)

temporal_coverage

Timedelta('52 days 23:04:11.134000')

### Finding

The lower-layer telemetry dataset contains an explicit `timestamp` field with
31,550 distinct values across 45,244 records.

No missing, non-positive, or infinite timestamp values were detected.

The timestamp values are consistent with Unix epoch time expressed in
milliseconds.

The observed temporal coverage extends approximately from
2025-05-09 to 2025-07-01 UTC, providing about 52 days of timestamped telemetry.

This temporal information makes the lower-layer dataset suitable for later
cross-source temporal alignment with other SRC-001 artifacts.

## 7. Leakage and Evaluation Risks

Identify fields and dataset characteristics that could introduce information
leakage or bias future evaluation results.

In [36]:
leakage_risk_fields = ground_truth_fields.copy()

leakage_risk_fields

['traffic_type', 'attack_category', 'attack_subcategory']

In [37]:
temporal_split_available = bool(temporal_fields)

temporal_split_available

True

### Finding

The primary leakage risk comes from the hierarchical ground-truth fields
identified in the telemetry dataset. These fields must be excluded appropriately
from predictive inputs when one of them is selected as the target.

No exact duplicate records were detected, reducing the risk of identical
observations appearing across evaluation partitions.

Unlike `Network_Dataset.csv`, the lower-layer dataset preserves explicit
timestamps. Therefore, chronological partitioning can be used in future
experiments to reduce temporal leakage.

Overall, the dataset provides sufficient structure to control the main
evaluation leakage risks identified during this audit.

## 8. Audit Findings

The audit of `Lower_Layer_Data.db` produced the following findings:

- The database contains one table, `lower_layer_data`, with 45,244 records and
  25 fields.
- The dataset contains lower-layer radio telemetry, UE/cell identifiers,
  timestamps, and hierarchical ground-truth labels.
- Missing values were detected only in `rnti`: 1,540 records
  (approximately 3.40%).
- No exact duplicate records were detected.
- Four constant fields were identified: `ri`, `cellid`, `in_sync`, and `pmi`.
- No infinite values, negative byte counters, or structurally invalid
  timestamps were detected.
- Ground-truth information is represented through `traffic_type`,
  `attack_category`, and `attack_subcategory`.
- Attack traffic represents approximately 76.43% of the dataset, while benign
  traffic represents approximately 23.57%.
- The dataset contains six high-level traffic/attack categories and
  19 subcategories.
- An explicit timestamp field is available and contains no missing,
  non-positive, or infinite values.
- The timestamps are consistent with Unix epoch milliseconds and cover
  approximately 53 days, from 2025-05-09 to 2025-07-01 UTC.
- The preserved temporal information supports future chronological evaluation
  and cross-source temporal alignment.
- Ground-truth fields remain the main potential source of target leakage and
  must be excluded appropriately from predictive features.

No data was modified or removed during this audit.

## 9. Lower-Layer Dataset Suitability Decision

`Lower_Layer_Data.db` is suitable for use in the DAICP MVP as a lower-layer
telemetry and temporal evidence source.

### Intended Use

The dataset will be used for:

- lower-layer radio and traffic telemetry analysis;
- scenario-level ground-truth reference;
- temporal correlation with other SRC-001 artifacts;
- chronological evaluation when temporal ordering is required;
- supporting RCA hypotheses with radio-layer evidence.

### Limitations

The dataset should not be treated as a standalone RCA source because:

- `rnti` contains approximately 3.40% missing values;
- `ri`, `cellid`, `in_sync`, and `pmi` are constant in the current subset;
- the class distribution is imbalanced;
- ground-truth fields must be excluded appropriately to prevent target leakage;
- the dataset represents only the lower-layer perspective of an incident.

### Decision

`Lower_Layer_Data.db` is accepted for the DAICP MVP.

It provides useful lower-layer telemetry, explicit timestamps, and scenario
labels that complement the network-layer evidence already identified in
`Network_Dataset.csv`.

The timestamp information is particularly valuable because it may allow
cross-source temporal alignment with PCAP and scenario telemetry artifacts.

SRC-001 cannot yet be considered fully suitable for the RCA workflow until the
remaining PCAP and TXT scenario files are audited and their consistency with
the CSV and lower-layer database is verified.